In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [2]:
# --- BƯỚC 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH ---
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import os


# --- BƯỚC 2: LOAD DỮ LIỆU TRÊN KAGGLE ---
# Lưu ý: Đảm bảo bạn đã click "Add Data" và add dataset "Brazilian E-Commerce Public Dataset by Olist" vào notebook.
# Đường dẫn mặc định trên Kaggle thường là:
data_folder = "/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce" 

FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

print("\n📥 Loading dữ liệu từ Kaggle Input...")
dfs = {}
for key, filename in FILES.items():
    filepath = os.path.join(data_folder, filename)
    if os.path.exists(filepath):
        dfs[key] = pd.read_csv(filepath)
        print(f"  ✅ Loaded {key}: {len(dfs[key])} rows")
    else:
        print(f"  ❌ Missing {filename} - Vui lòng kiểm tra lại đường dẫn Add Data")




📥 Loading dữ liệu từ Kaggle Input...
  ✅ Loaded orders: 99441 rows
  ✅ Loaded customers: 99441 rows
  ✅ Loaded items: 112650 rows
  ✅ Loaded payments: 103886 rows
  ✅ Loaded reviews: 99224 rows
  ✅ Loaded products: 32951 rows
  ✅ Loaded sellers: 3095 rows
  ✅ Loaded geolocation: 1000163 rows
  ✅ Loaded translation: 71 rows


In [3]:
# --- BƯỚC 3: TIỀN XỬ LÝ & JOIN DỮ LIỆU (CẬP NHẬT) ---
print("\n🔧 Bắt đầu tiền xử lý và khắc phục lỗi join...")

# 1. Xử lý geolocation: lấy tọa độ trung bình theo zip_code
df_geo_clean = dfs["geolocation"].groupby("geolocation_zip_code_prefix", as_index=False).agg(
    lat_avg=("geolocation_lat", "mean"),
    lng_avg=("geolocation_lng", "mean")
).round(4)

# 2. XỬ LÝ GOM NHÓM (AGGREGATE) TRƯỚC KHI JOIN ĐỂ TRÁNH TRÙNG LẶP
# Bảng Payments: Một đơn có thể thanh toán nhiều lần/nhiều thẻ -> Tính tổng tiền
df_payments_agg = dfs["payments"].groupby("order_id", as_index=False).agg(
    total_payment_value=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max")
)

# Bảng Reviews: Một đơn có thể có nhiều review -> Lấy điểm trung bình và comment đầu tiên
df_reviews_agg = dfs["reviews"].groupby("order_id", as_index=False).agg(
    review_score=("review_score", "mean"),
    review_comment_message=("review_comment_message", "first")
)

# 3. JOIN CÁC BẢNG (MASTER JOIN) SỬ DỤNG THAM SỐ SUFFIXES
df_master = dfs["orders"]
df_master = df_master.merge(dfs["customers"], on="customer_id", how="left", suffixes=('', '_drop'))
df_master = df_master.merge(
    df_geo_clean, 
    left_on="customer_zip_code_prefix", 
    right_on="geolocation_zip_code_prefix", 
    how="left",
    suffixes=('', '_drop')
).drop(columns=["geolocation_zip_code_prefix"])

# Join với bảng items, products, sellers (số dòng sẽ tăng theo số lượng item thực tế trong đơn)
df_master = df_master.merge(dfs["items"], on="order_id", how="left", suffixes=('', '_drop'))
df_master = df_master.merge(dfs["products"], on="product_id", how="left", suffixes=('', '_drop'))
df_master = df_master.merge(dfs["translation"], on="product_category_name", how="left", suffixes=('', '_drop'))
df_master = df_master.merge(dfs["sellers"], on="seller_id", how="left", suffixes=('', '_drop'))

# Join với các bảng đã được gom nhóm (Payments & Reviews)
df_master = df_master.merge(df_payments_agg, on="order_id", how="left", suffixes=('', '_drop'))
df_master = df_master.merge(df_reviews_agg, on="order_id", how="left", suffixes=('', '_drop'))
print(df_master.columns.tolist())



🔧 Bắt đầu tiền xử lý và khắc phục lỗi join...
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'lat_avg', 'lng_avg', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'total_payment_value', 'payment_installments_max', 'review_score', 'review_comment_message']


In [4]:
# 4. LOẠI BỎ NHANH CÁC CỘT TRÙNG LẶP
df_master = df_master.loc[:, ~df_master.columns.str.endswith('_drop')]

print(f"  ✅ Đã join bảng an toàn. Tổng bản ghi hiện tại: {len(df_master)}")

  ✅ Đã join bảng an toàn. Tổng bản ghi hiện tại: 113425


In [5]:
print(df_master.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'lat_avg', 'lng_avg', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'total_payment_value', 'payment_installments_max', 'review_score', 'review_comment_message']


In [6]:
# --- BƯỚC 4 & 5: CHUẨN HÓA CỘT VÀ XỬ LÝ MISSING VALUES (CẬP NHẬT) ---
print("\n🧹 Đang chuẩn hóa cột và xử lý missing values...")

# 1. Chuyển toàn bộ tên cột về in thường và thay khoảng trắng
df_master.columns = df_master.columns.str.lower().str.strip().str.replace(' ', '_')

# 2. Xử lý Missing Values chuẩn xác
# Text comment thì fill "No Comment" để an toàn cho NLP Pipeline sau này
df_master["review_comment_message"] = df_master["review_comment_message"].fillna("No Comment")

# QUAN TRỌNG: Giữ nguyên NaN cho review_score, KHÔNG dùng .fillna(0)
# Việc dropna(subset=['review_score']) sẽ do Member 2 (ML Engineer) tự xử lý 
# trên tập training của họ. Giữ NaN ở Master Data để các bạn làm Clustering/RFM 
# không bị mất đi dữ liệu đơn hàng (order) giá trị.

# 3. Xử lý thời gian và Feature Engineering
timestamp_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date", "review_creation_date",
    "review_answer_timestamp", "shipping_limit_date"
]

for col in timestamp_cols:
    if col in df_master.columns:
        df_master[col] = pd.to_datetime(df_master[col], errors="coerce")

# Tính toán độ trễ giao hàng
if "order_delivered_customer_date" in df_master.columns and "order_estimated_delivery_date" in df_master.columns:
    df_master["delivery_delay_days"] = (df_master["order_delivered_customer_date"] - df_master["order_estimated_delivery_date"]).dt.days

print(f"  ✅ Đã xử lý xong. Số lượng cột hiện tại: {len(df_master.columns)}")


🧹 Đang chuẩn hóa cột và xử lý missing values...
  ✅ Đã xử lý xong. Số lượng cột hiện tại: 37


In [7]:
print(df_master.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'lat_avg', 'lng_avg', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'total_payment_value', 'payment_installments_max', 'review_score', 'review_comment_message', 'delivery_delay_days']


In [8]:
# --- BƯỚC 6: CHIA TẬP DỮ LIỆU TRAIN / TEST ---

# Chia dữ liệu theo tỷ lệ 80% Train - 20% Test
# random_state=42 cực kỳ quan trọng để đảm bảo mọi người trong nhóm 
# đều nhận được kết quả chia cắt giống hệt nhau khi chạy lại code.
df_train, df_test = train_test_split(df_master, test_size=0.2, random_state=42)

print(f"  • Kích thước tập Train: {df_train.shape}")
print(f"  • Kích thước tập Test: {df_test.shape}")

  • Kích thước tập Train: (90740, 37)
  • Kích thước tập Test: (22685, 37)


In [9]:
# --- BƯỚC 6: KIỂM TRA KẾT QUẢ ---
print("\n📋 Kết quả sau tiền xử lý:")
print(f"  • Tổng columns: {len(df_master.columns)}")
print(f"  • Tổng rows: {len(df_master)}")


# --- BƯỚC 6: LƯU DỮ LIỆU ĐÃ XỬ LÝ VÀO KAGGLE WORKING ---
print("\n💾 Đang chuẩn bị lưu dữ liệu...")

# Lưu vào /kaggle/working/ để file được giữ lại sau khi chạy xong (Output)
SAVE_PATH = "/kaggle/working/processed_data/week1"
os.makedirs(SAVE_PATH, exist_ok=True)

df_master.to_parquet(f"{SAVE_PATH}/master_dataset.parquet", index=False)
print(f"  ✅ Đã lưu dataset vào: {SAVE_PATH}/master_dataset.parquet")

with open(f"{SAVE_PATH}/schema_info.txt", "w") as f:
    df_master.info(buf=f)
print(f"  ✅ Đã lưu schema vào: {SAVE_PATH}/schema_info.txt")
# Xuất ra định dạng Parquet để giữ nguyên Data Types (nhanh và nhẹ hơn CSV)
df_train.to_parquet(f"{SAVE_PATH}/train_data.parquet", index=False)
df_test.to_parquet(f"{SAVE_PATH}/test_data.parquet", index=False)

# Tùy chọn: Xuất thêm một bản CSV (sample 1000 dòng) để các bạn làm UI/DA 
# có thể mở nhanh bằng Excel để xem cấu trúc[cite: 71, 72].
df_train.sample(1000, random_state=42).to_csv(f"{SAVE_PATH}/sample_train_1k.csv", index=False)

print(f"  ✅ Đã xuất toàn bộ file vào thư mục: {SAVE_PATH}")
print("  🚀 Bạn đã có thể tải các file này về và gửi cho team (hoặc push lên GitHub)!")


📋 Kết quả sau tiền xử lý:
  • Tổng columns: 37
  • Tổng rows: 113425

💾 Đang chuẩn bị lưu dữ liệu...
  ✅ Đã lưu dataset vào: /kaggle/working/processed_data/week1/master_dataset.parquet
  ✅ Đã lưu schema vào: /kaggle/working/processed_data/week1/schema_info.txt
  ✅ Đã xuất toàn bộ file vào thư mục: /kaggle/working/processed_data/week1
  🚀 Bạn đã có thể tải các file này về và gửi cho team (hoặc push lên GitHub)!


In [10]:
# --- IV.2.1. RFM ANALYSIS ---
import pandas as pd
from datetime import timedelta

print("📊 Đang bắt đầu phân tích RFM...")

# 1. Thiết lập ngày tham chiếu (Reference Date) 
# Lấy ngày cuối cùng trong dữ liệu cộng thêm 1 ngày để tính Recency
reference_date = df_master['order_purchase_timestamp'].max() + timedelta(days=1)

# 2. Tính toán các giá trị R, F, M theo từng khách hàng (customer_unique_id)
rfm = df_master.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days, # Recency
    'order_id': 'nunique',                                               # Frequency
    'total_payment_value': 'sum'                                         # Monetary
})

# 3. Đổi tên cột cho đúng chuẩn RFM
rfm.rename(columns={
    'order_purchase_timestamp': 'Recency',
    'order_id': 'Frequency',
    'total_payment_value': 'Monetary'
}, inplace=True)

# 4. Hiển thị bảng thống kê mô tả (.describe()) để đưa vào báo cáo
print("\n📋 Bảng thống kê mô tả RFM (Dùng số liệu này điền vào bảng báo cáo):")
rfm_desc = rfm.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
display(rfm_desc)

# 5. Lưu kết quả RFM ra file để phục vụ bài toán Clustering (Tuần 3)
# 5. Lưu kết quả RFM ra file Parquet
SAVE_PATH = "/kaggle/working/processed_data/week1"
os.makedirs(SAVE_PATH, exist_ok=True)

rfm_file_path = f"{SAVE_PATH}/rfm_dataset.parquet"
rfm.to_parquet(rfm_file_path, index=False)

print(f"\n✅ Đã lưu kết quả RFM Analysis thành công tại: {rfm_file_path}")

📊 Đang bắt đầu phân tích RFM...

📋 Bảng thống kê mô tả RFM (Dùng số liệu này điền vào bảng báo cáo):


,count,mean,std,min,25%,50%,75%,max
Recency,96096.0,288.735691,153.414676,1.0,164.00,269.00,398.0000,773.00
Frequency,96096.0,1.034809,0.214384,1.0,1.00,1.00,1.0000,17.00
Monetary,96096.0,213.023712,640.917083,0.0,63.99,113.15,202.7325,109312.64



✅ Đã lưu kết quả RFM Analysis thành công tại: /kaggle/working/processed_data/week1/rfm_dataset.parquet
